# Universal Player Classification (Outfield & Goalkeepers)

In this notebook, we classify all players into distinct tactical roles based on their statistical profiles using **Statistical Normalization (Percentiles)**.

By comparing each player against their positional peers in the dataset, we calculate a score from 0 to 100 for each relevant tactical role. This makes the classification robust, scalable to any league, and perfectly suited for a Big Data scouting search engine.

In [17]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load datasets
df_outfield = pd.read_csv('../data/clean/clean_df_outfield.csv')
df_gk = pd.read_csv('../data/clean/clean_df_gk.csv')
print(f"Total outfield players loaded: {len(df_outfield)}")
print(f"Total goalkeepers loaded: {len(df_gk)}")

Total outfield players loaded: 772
Total goalkeepers loaded: 83


In [18]:
# Part 1: Outfield Classification Configuration
outfield_config = {
    'Centre-Back': {
        'Positions': ['Centre-Back'],
        'Aerial Stopper': ['Aerial Duels Won (Per90)', 'Clearances (Per90)'],
        'Ball Playing Defender': ['Pass Completion Rate', 'Successful Passes (Per90)', 'Key Passes (Per90)'],
        'Defensive Sweeper': ['Interceptions (Per90)', 'Tackles (Per90)', 'Shots Blocked (Per90)', 'Ground Duels Won (Per90)']
    },
    'Full-Back': {
        'Positions': ['Left-Back', 'Right-Back'],
        'Wing Back': ['Successful Crosses (Per90)', 'Key Passes (Per90)', 'Expected Assists (xA) (Per90)', 'Successful Dribbles (Per90)'],
        'Inverted Full Back': ['Successful Passes (Per90)', 'Pass Completion Rate', 'Ground Duels Won (Per90)'],
        'Defensive Full Back': ['Tackles (Per90)', 'Interceptions (Per90)', 'Clearances (Per90)']
    },
    'Defensive Midfield': {
        'Positions': ['Defensive Midfield'],
        'Anchor Man': ['Interceptions (Per90)', 'Clearances (Per90)', 'Shots Blocked (Per90)'],
        'Deep Lying Playmaker': ['Successful Passes (Per90)', 'Pass Completion Rate', 'Key Passes (Per90)'],
        'Ball Winning Midfielder': ['Tackles (Per90)', 'Ground Duels Won (Per90)']
    },
    'Central Midfield': {
        'Positions': ['Central Midfield'],
        'Box to Box': ['Ground Duels Won (Per90)', 'Successful Passes (Per90)', 'Shots Taken (Per90)', 'Interceptions (Per90)'],
        'Mezzala': ['Key Passes (Per90)', 'Successful Dribbles (Per90)', 'Expected Goals (xG) (Per90)', 'Expected Assists (xA) (Per90)'],
        'Ball Winning Midfielder': ['Tackles (Per90)', 'Interceptions (Per90)', 'Ground Duels Won (Per90)']
    },
    'Attacking Midfield': {
        'Positions': ['Attacking Midfield'],
        'Shadow Striker': ['Goals Scored (Per90)', 'Expected Goals (xG) (Per90)', 'Shots On Target (Per90)'],
        'Playmaker': ['Key Passes (Per90)', 'Expected Assists (xA) (Per90)', 'Successful Passes (Per90)']
    },
    'Winger': {
        'Positions': ['Left Winger', 'Right Winger', 'Left Midfield'],
        'Inside Forward': ['Goals Scored (Per90)', 'Shots Taken (Per90)', 'Expected Goals (xG) (Per90)', 'Successful Dribbles (Per90)'],
        'Traditional Winger': ['Successful Crosses (Per90)', 'Key Passes (Per90)'],
        'Balanced Winger': ['Expected Assists (xA) (Per90)', 'Successful Dribbles (Per90)', 'Successful Passes (Per90)']
    },
    'Striker': {
        'Positions': ['Centre-Forward', 'Second Striker'],
        'Poacher': ['Goals Scored (Per90)', 'Shot Conversion Rate', 'Shots On Target (Per90)'],
        'Target Man': ['Aerial Duels Won (Per90)', 'Ground Duels Won (Per90)', 'Shots Taken (Per90)'],
        'False Nine': ['Key Passes (Per90)', 'Expected Assists (xA) (Per90)', 'Successful Dribbles (Per90)', 'Successful Passes (Per90)']
    }
}

# Part 2: Goalkeeper Classification Configuration
gk_config = {
    'Goalkeeper': {
        'Positions': ['Goalkeeper'],
        'Shot Stopper': ['Saves (Per90)', 'Save Percentage', 'Punches (Per90)'],
        'Sweeper Keeper': ['Interceptions (Per90)', 'Clearances (Per90)', 'Pass Completion Rate', 'Successful Passes (Per90)']
    }
}

print("Configuration mapping successfully loaded for all positions.")

Configuration mapping successfully loaded for all positions.


In [19]:
# Universal Classification Engine Function
def classify_players(df, role_config):
    all_processed_dfs = []
    for broad_pos, config in role_config.items():
        target_positions = config['Positions']
        role_map = {k: v for k, v in config.items() if k != 'Positions'}
            
        pos_df = df[df['Position'].isin(target_positions)].copy()
        if len(pos_df) == 0:
            continue
            
        all_features = set(feature for features in role_map.values() for feature in features)
        
        for feature in all_features:
            if feature in pos_df.columns:
                median_val = pos_df[feature].median()
                pos_df[f'{feature}_Pct'] = pos_df[feature].fillna(median_val).rank(pct=True)
            else:
                pos_df[f'{feature}_Pct'] = 0.5 
                
        for role, features in role_map.items():
            pct_cols = [f'{feature}_Pct' for feature in features]
            pos_df[f'{role}_Score'] = pos_df[pct_cols].mean(axis=1) * 100
            
        def assign_best_role(row):
            scores = {role: row[f'{role}_Score'] for role in role_map.keys()}
            return max(scores, key=scores.get)
            
        pos_df['Tactical_Role'] = pos_df.apply(assign_best_role, axis=1)
        pos_df['Broad_Position'] = broad_pos
        all_processed_dfs.append(pos_df)
        
    return pd.concat(all_processed_dfs, ignore_index=True)

# Run the engine for both datasets
final_outfield_df = classify_players(df_outfield, outfield_config)
final_gk_df = classify_players(df_gk, gk_config)

print("Classification process completed for Outfield Players and Goalkeepers.")

Classification process completed for Outfield Players and Goalkeepers.


In [20]:
# Analyze the role distribution
print("Outfield Tactical Role Distribution:\n")
print(final_outfield_df.groupby(['Broad_Position', 'Tactical_Role']).size().unstack(fill_value=0))
print("\n" + "="*70 + "\n")

print("Goalkeeper Tactical Role Distribution:\n")
print(final_gk_df.groupby(['Broad_Position', 'Tactical_Role']).size().unstack(fill_value=0))
print("\n" + "="*70 + "\n")

Outfield Tactical Role Distribution:

Tactical_Role       Aerial Stopper  Anchor Man  Balanced Winger  \
Broad_Position                                                    
Attacking Midfield               0           0                0   
Central Midfield                 0           0                0   
Centre-Back                     55           0                0   
Defensive Midfield               0          21                0   
Full-Back                        0           0                0   
Striker                          0           0                0   
Winger                           0           0               36   

Tactical_Role       Ball Playing Defender  Ball Winning Midfielder  \
Broad_Position                                                       
Attacking Midfield                      0                        0   
Central Midfield                        0                       44   
Centre-Back                            59                        0   
Defensiv

In [21]:
# Save the classified data for the Scout Search Engine
final_outfield_df.to_csv('../data/clean/classified_outfield_players.csv', index=False)
final_gk_df.to_csv('../data/clean/classified_goalkeepers.csv', index=False)
print("Data successfully exported to 'classified_outfield_players.csv' and 'classified_goalkeepers.csv'.")

Data successfully exported to 'classified_outfield_players.csv' and 'classified_goalkeepers.csv'.
